# Module 1.3: DuckDB & Parquet for Industrial SCADA Historian Archives
### 🇵🇱 DuckDB i Parquet w Archiwizacji Danych Historiana SCADA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m01_l03_duckdb_parquet_industrial_archives.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Execute blazingly fast SQL aggregations over multi-gigabyte partitioned Parquet directories directly without spinning up heavy database servers.  
> 🇵🇱 **Streszczenie:** Wykonuj błyskawiczne zapytania SQL na wielogigabajtowych archiwach Parquet bez konieczności instalowania ciężkich serwerów bazodanowych.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Industrial Historian Storage**: Traditional CSV or unindexed RDBMS databases choke on multi-year time-series telemetry exceeding hundreds of gigabytes.
2. **Apache Parquet Benefits**:
   - Columnar projection: Only requested sensor columns are read from disk.
   - Snappy/Zstandard compression: Slashes disk footprints by $70\%-85\%$.
   - Chunk statistics: Min/max values per row-group enable predicate pushdown, skipping irrelevant days/months entirely.
3. **DuckDB In-Process Engine**: Runs zero-copy vectorized SQL queries directly against compressed `.parquet` files without loading datasets into Python RAM.


In [ ]:
# 1. Create Compressed Industrial Parquet Archive
import tempfile
import pandas as pd

np.random.seed(101)
n_records = 300_000
temp_dir = tempfile.mkdtemp()
parquet_path = f"{temp_dir}/sensor_archive.parquet"

df_archive = pd.DataFrame({
    "timestamp": pd.date_range("2026-01-01", periods=n_records, freq="1s"),
    "machine_id": np.random.choice(["KILN_A", "KILN_B", "FURNACE_01"], size=n_records),
    "zone_temp_c": np.random.normal(850.0, 25.0, size=n_records),
    "furnace_pressure_mbar": np.random.normal(1013.2, 8.5, size=n_records),
    "heater_duty_pct": np.random.uniform(40.0, 95.0, size=n_records)
})

# Save to Parquet with Snappy compression
df_archive.to_parquet(parquet_path, engine="pyarrow", compression="snappy", index=False)
parquet_size_mb = os.path.getsize(parquet_path) / (1024 * 1024)
print(f"✓ Saved {n_records:,} industrial records to Parquet ({parquet_size_mb:.2f} MB).")


In [ ]:
# 2. Query Parquet Archive using DuckDB SQL
try:
    import duckdb
    con = duckdb.connect()
    # Direct SQL query over Parquet file without loading into memory
    query = f"""
        SELECT 
            machine_id,
            COUNT(*) AS total_samples,
            ROUND(AVG(zone_temp_c), 2) AS avg_temp,
            ROUND(MAX(zone_temp_c), 2) AS max_temp,
            ROUND(AVG(furnace_pressure_mbar), 2) AS avg_pressure,
            ROUND(STDDEV(zone_temp_c), 2) AS temp_std
        FROM '{parquet_path}'
        WHERE heater_duty_pct > 80.0
        GROUP BY machine_id
        ORDER BY machine_id
    """
    result_df = con.execute(query).df()
    print("DuckDB Query Results (High Duty Cycle Operation):")
    print(result_df)
    
    # Plot DuckDB aggregated stats
    plt.figure(figsize=(9, 4))
    plt.bar(result_df['machine_id'], result_df['avg_temp'], yerr=result_df['temp_std'], 
            capsize=5, color='#0284c7', alpha=0.85, width=0.45)
    plt.title("Furnace High-Duty Average Temperature & Stability (DuckDB SQL)", fontweight='bold')
    plt.ylabel("Temperature [°C]")
    plt.ylim(800, 900)
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("DuckDB not installed; using Pandas query equivalent.")


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Partition plant telemetry into hierarchical Parquet year/month/machine folders *(🇵🇱 Podziel dane telemetrii na hierarchiczne partycje Parquet (rok/miesiąc/maszyna))*
- [ ] **Task 2:** Query partitioned Parquet files directly using DuckDB vectorized SQL *(🇵🇱 Wykonaj analityczne zapytanie SQL bezpośrednio na plikach Parquet za pomocą DuckDB)*
- [ ] **Task 3:** Compute time-weighted rolling machine availability (OEE) metrics in SQL *(🇵🇱 Oblicz wskaźnik dostępności maszyny (OEE) w oknie czasowym z poziomu SQL)*


In [ ]:
# Exercise Challenge:
# Run a query to find the peak furnace pressure recorded during the entire archive
max_pressure = df_archive['furnace_pressure_mbar'].max()
print(f"Peak Archive Pressure: {max_pressure:.2f} mbar")
assert max_pressure > 1030.0, "Unexpected peak pressure value."
print("✓ Parquet archive analytical query passed.")
